<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB15_Transfer_Learning_Feature_Extraction_vs_Fine_Tuning_ES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **NB15 · Clase 15 — Transfer Learning: extracción de características frente a fine-tuning**

## Bloque 3: IA — Deep Learning (continuación)

`NB13` entrenó una CNN completamente desde cero con ~400 imágenes submarinas reales — un dataset pequeño para los estándares de deep learning, donde `sus filtros tuvieron que aprenderlo todo, incluidos los bordes básicos y las texturas, solo a partir de esa muestra pequeña`. Esta clase se pregunta: ¿y si no empezáramos desde cero? El **transfer learning** reutiliza una red ya entrenada con millones de fotos no relacionadas, bajo la suposición razonable (respaldada por la referencia final de `NB13` a Zeiler & Fergus) de que sus filtros iniciales, genéricos, son útiles para casi cualquier tarea de imagen. Comparamos las dos estrategias estándar — **extracción de características** y **fine-tuning** — directamente entre sí, y frente al resultado desde cero de `NB13`, sobre la misma tarea real de clasificación de crecimiento marino de LIACi.

### Objetivos de aprendizaje

Al terminar esta clase, el alumnado será capaz de:
- Explicar qué es el transfer learning y por qué funciona, en términos de qué aprende cada capa de una CNN.
- Distinguir la extracción de características (backbone congelado) del fine-tuning (backbone parcialmente descongelado), y explicar el compromiso entre ambos.
- Cargar y preprocesar correctamente datos para un modelo preentrenado real (`ResNet18` de `torchvision`).
- Congelar y descongelar selectivamente capas de un modelo PyTorch preentrenado.
- Comparar de forma justa el entrenamiento desde cero, la extracción de características y el fine-tuning, sobre los mismos datos y evaluación reales.

### Agenda (clase de 2 horas)

| # | Segmento de la clase | Duración aprox. | Tipo |
|---|---------------------|:---:|:---:|
| 1 | Repaso, hoja de ruta de hoy | 5 min | Teoría |
| 2 | ¿Qué es el transfer learning y por qué funciona? | 10 min | Teoría |
| 3 | Dos estrategias: extracción de características frente a fine-tuning | 15 min | Teoría + Práctica |
| 4 | Cargar un modelo preentrenado real | 15 min | Práctica |
| 5 | Dataset real: reutilizar el pipeline (corregido) de LIACi de NB13 | 15 min | Práctica |
| 6 | Preprocesado para un modelo preentrenado | 10 min | Teoría + Práctica |
| 7 | Práctica: extracción de características | 15 min | Práctica |
| 8 | Práctica: fine-tuning | 15 min | Práctica |
| 9 | Comparar los tres enfoques | 15 min | Práctica |
| 10 | Resumen, tarea, próxima clase | 5 min | Teoría |

> Los tiempos son orientación aproximada, no un guion estricto — no hay descansos programados. Si terminamos todo con tiempo de sobra, la clase acaba antes; eso puede pasar y no pasa nada.

> **Antes de empezar**: igual que `NB13`, esta clase descarga el dataset real de LIACi (~1 GB) y entrena con imágenes reales. Cambia ahora a un entorno de ejecución con GPU si está disponible (`Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`).

---

## 1. Repaso: dónde estamos

- **`NB13`**: una CNN entrenada desde cero con imágenes submarinas reales — cada filtro aprendido solo a partir de ~400 ejemplos.
- **`NB14`**: teoría de RNN/LSTM y un modelo real de previsión de secuencias.
- **`NB15`** (hoy): reutilizar una red ya entrenada con millones de imágenes, en vez de partir de cero.

---

## 2. ¿Qué es el transfer learning y por qué funciona?

El **[transfer learning](https://en.wikipedia.org/wiki/Transfer_learning)** toma un modelo entrenado en una tarea (normalmente un dataset enorme y general) y reutiliza parte o todos sus pesos aprendidos como punto de partida para una tarea distinta, normalmente más pequeña — en vez de inicializar cada peso al azar, como hacía el `HullCNN` de `NB13`.

Por qué esto funciona específicamente para las CNN, enlazando directamente con la visualización de mapas de características del §9 de `NB13`: una red entrenada con **ImageNet** (1,4 millones de fotos naturales en 1000 categorías — gatos, coches, muebles, todo menos imágenes de cascos submarinos) sigue aprendiendo, en sus capas iniciales, `filtros que detectan bordes, gradientes de color y texturas simples`. Eso es útil para reconocer *casi cualquier* patrón visual, incluidas las condiciones de un casco submarino — solo las capas posteriores, más específicas de la tarea, necesitan cambiar de verdad. Reutilizar las capas iniciales significa que nuestro pequeño dataset de ~400 imágenes solo tiene que enseñarle a la red qué es *distinto* en nuestra tarea, no cómo ver en absoluto.

---

## 3. Dos estrategias: extracción de características frente a fine-tuning

| | Extracción de características | [Fine-tuning](https://en.wikipedia.org/wiki/Fine-tuning_%28deep_learning%29) |
|---|---|---|
| Pesos del backbone | **Congelados** — nunca se actualizan | **Parcialmente descongelados** — algunas capas siguen entrenando |
| Qué se entrena | Solo una nueva capa final (la cabeza clasificadora) | La nueva cabeza **y** algunas de las capas finales del backbone |
| Velocidad de entrenamiento | Rápida — la mayor parte de la red no hace pasada hacia atrás | Más lenta — se actualizan más parámetros |
| Datos necesarios | Funciona con muy pocos datos | Necesita algo más de datos, o tasas de aprendizaje cuidadosas, para evitar el sobreajuste o el "olvido" |
| Cuándo preferirla | Tu tarea se parece a los datos de preentrenamiento, o tu dataset es diminuto | Tu tarea difiere más de los datos de preentrenamiento, y tienes datos suficientes para adaptar con seguridad más parte de la red |

Visualicemos qué capas toca cada estrategia:

Dibujemos ambas estrategias una junto a otra, bloques congelados frente a entrenables:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

block_labels = ["conv\nblock 1", "conv\nblock 2", "conv\nblock 3", "conv\nblock 4", "avgpool", "fc\n(new)"]
feature_extraction_frozen = [True, True, True, True, True, False]
fine_tuning_frozen = [True, True, True, False, False, False]

for ax, title, frozen_pattern in zip(
    axes, ["Feature extraction", "Fine-tuning"], [feature_extraction_frozen, fine_tuning_frozen]
):
    for i, (label, frozen) in enumerate(zip(block_labels, frozen_pattern)):
        color = "lightgray" if frozen else "lightcoral"
        ax.add_patch(patches.Rectangle((i, 0), 0.9, 1, facecolor=color, edgecolor="black"))
        ax.text(i + 0.45, 0.5, label, ha="center", va="center", fontsize=8)
        ax.text(i + 0.45, -0.3, "frozen" if frozen else "trainable", ha="center", fontsize=7,
                color="gray" if frozen else "darkred")
    ax.set_xlim(-0.3, len(block_labels))
    ax.set_ylim(-0.6, 1.3)
    ax.axis("off")
    ax.set_title(title)

plt.tight_layout()
plt.show()

El fine-tuning siempre descongela las capas *finales* (más cerca de la salida), nunca las primeras — `esos filtros iniciales, de bordes y texturas más genéricos, son precisamente la parte en la que más confía el transfer learning`.

---

## 4. Cargar un modelo preentrenado real

`torchvision` incluye varios modelos preentrenados con ImageNet. Usaremos **ResNet18** — una arquitectura real y muy utilizada, lo bastante pequeña para hacerle fine-tuning rápido en CPU si hace falta:

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision.models import ResNet18_Weights

weights = ResNet18_Weights.DEFAULT
resnet = models.resnet18(weights=weights)

print(resnet.fc)  # the final layer -- built for ImageNet's 1,000 classes
sum(p.numel() for p in resnet.parameters())

Ese número de parámetros empequeñece al `HullCNN` de `NB13` — pero estamos a punto de congelar casi todos, así que el número *efectivo* de pesos que entrenaremos nosotros mismos acabará siendo mucho menor, no mayor.

**Pruébalo tú mismo**: `layer4`, en las Secciones 7-8, se refiere a uno de varios bloques con nombre. Lístalos para ver la estructura completa disponible para congelar o descongelar:

In [ ]:
for name, _ in resnet.named_children():
    print(name)


---

## 5. Dataset real: reutilizar el pipeline (corregido) de LIACi de `NB13`

Mismo dataset real, misma condición objetivo (crecimiento marino), misma lógica de construcción de etiquetas que `NB13` — incluida la corrección para los ficheros de máscara `.bmp` de LIACi (que no comparten la extensión `.jpg` de las imágenes fuente):

In [ ]:
!wget -q -O liaci_data.zip https://liaci.sintef.cloud/download_data/data.zip
!unzip -oq liaci_data.zip

import os
from PIL import Image
import numpy as np

base_dir = "LIACi_dataset_pretty"
imgs_path = os.path.join(base_dir, "images")
masks_path = os.path.join(base_dir, "masks")

image_files = sorted(os.listdir(imgs_path))
target_class = "marine_growth"

def label_for(fname, cls):
    stem = os.path.splitext(fname)[0]
    mask_file = os.path.join(masks_path, cls, stem + ".bmp")
    if not os.path.exists(mask_file):
        return 0
    mask = np.array(Image.open(mask_file).convert("L"))
    return int(mask.max() > 0)

all_labels = {f: label_for(f, target_class) for f in image_files}
print(f"{sum(all_labels.values())} / {len(all_labels)} images show '{target_class}'")

Submuestra equilibrada, exactamente igual que en `NB13`:

In [ ]:
import random

random.seed(42)
positive_files = [f for f in image_files if all_labels[f] == 1]
negative_files = [f for f in image_files if all_labels[f] == 0]

n_per_class = min(len(positive_files), len(negative_files), 200)
sample_files = random.sample(positive_files, n_per_class) + random.sample(negative_files, n_per_class)
random.shuffle(sample_files)
print("Sample size:", len(sample_files))

---

## 6. Preprocesado para un modelo preentrenado

`NB13` redimensionaba las imágenes a 96×96 y normalizaba los píxeles a 0-1 — una elección libre, ya que el `HullCNN` era nuestro y lo diseñamos a nuestro gusto. Un modelo **preentrenado** es distinto: `fue entrenado esperando que las imágenes se preprocesaran de una forma muy concreta` (típicamente 224×224, normalizadas con la media y desviación típica exactas por canal de ImageNet), y darle cualquier otra cosa degrada el rendimiento silenciosamente, porque cada valor de píxel quedaría en un rango distinto del que sus filtros congelados aprendieron a esperar.

La API moderna de `torchvision` guarda el preprocesado exacto que espera un conjunto de pesos dado, así que no tenemos que codificarlo nosotros a mano:

In [ ]:
preprocess = weights.transforms()
print(preprocess)

Usémosla directamente para construir nuestros tensores de imagen — los mismos ficheros reales, esta vez preprocesados correctamente para una red preentrenada:

In [ ]:
def load_and_preprocess(fname):
    img = Image.open(os.path.join(imgs_path, fname)).convert("RGB")
    return preprocess(img)

X_img = torch.stack([load_and_preprocess(f) for f in sample_files])
y_img = torch.tensor([all_labels[f] for f in sample_files], dtype=torch.float32).view(-1, 1)
X_img.shape

División 60/20/20 por índice, estratificada, igual que en toda tarea de clasificación desde `NB12`:

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

labels_arr = y_img.numpy().ravel()
idx = np.arange(len(labels_arr))
idx_train_full, idx_test = train_test_split(idx, test_size=0.2, random_state=42, stratify=labels_arr)
idx_train, idx_val = train_test_split(
    idx_train_full, test_size=0.25, random_state=42, stratify=labels_arr[idx_train_full]
)

X_train, y_train = X_img[idx_train], y_img[idx_train]
X_val, y_val = X_img[idx_val], y_img[idx_val]
X_test, y_test = X_img[idx_test], y_img[idx_test]
X_train.shape, X_val.shape, X_test.shape

---

## 7. Práctica: extracción de características

Congela todos los parámetros del backbone, y después sustituye la capa final por una nueva, dimensionada para nuestra tarea binaria — la nueva capa es entrenable por defecto, ya que nunca se le ha puesto `requires_grad` a `False`:

In [ ]:
fe_model = models.resnet18(weights=weights)

for param in fe_model.parameters():
    param.requires_grad = False

fe_model.fc = nn.Linear(fe_model.fc.in_features, 1)

trainable = sum(p.numel() for p in fe_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in fe_model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")

Solo la nueva cabeza es entrenable — compara esta cifra con el número de parámetros del `HullCNN` de `NB13`, de la Parte 8 de esa clase. Entrenemos con la misma receta que todo clasificador desde `NB11`:

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, fe_model.parameters()), lr=0.001)

n_epochs = 10
fe_train_losses, fe_val_losses = [], []

for epoch in range(n_epochs):
    fe_model.train()
    optimizer.zero_grad()
    loss = criterion(fe_model(X_train), y_train)
    loss.backward()
    optimizer.step()
    fe_train_losses.append(loss.item())

    fe_model.eval()
    with torch.no_grad():
        fe_val_losses.append(criterion(fe_model(X_val), y_val).item())

    print(f"Epoch {epoch + 1}/{n_epochs} - train: {fe_train_losses[-1]:.4f} - val: {fe_val_losses[-1]:.4f}")

Fíjate en que hemos necesitado muchas menos épocas que el entrenamiento desde cero de `NB13` — el backbone ya "sabe ver"; `la nueva cabeza solo tiene que aprender un límite simple sobre unas características que ya son significativas`.

---

## 8. Práctica: fine-tuning

Mismo punto de partida, pero esta vez descongelamos `layer4` (el último bloque residual de ResNet18, el más específico de la tarea) además de la nueva cabeza — igual que en el diagrama de la Parte 3:

In [ ]:
ft_model = models.resnet18(weights=weights)

for param in ft_model.parameters():
    param.requires_grad = False
for param in ft_model.layer4.parameters():
    param.requires_grad = True

ft_model.fc = nn.Linear(ft_model.fc.in_features, 1)

trainable = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in ft_model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")

Más parámetros entrenables que en la Parte 7, pero sigue siendo una fracción pequeña de toda la red. Un detalle importante: por convención, el fine-tuning usa una **tasa de aprendizaje menor** que el entrenamiento desde cero — estamos ajustando ligeramente unos pesos preentrenados que ya son buenos, no aprendiendo desde la nada, y actualizaciones grandes arriesgan destruir lo que ya habían aprendido:

In [ ]:
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, ft_model.parameters()), lr=0.0001)

ft_train_losses, ft_val_losses = [], []

for epoch in range(n_epochs):
    ft_model.train()
    optimizer.zero_grad()
    loss = criterion(ft_model(X_train), y_train)
    loss.backward()
    optimizer.step()
    ft_train_losses.append(loss.item())

    ft_model.eval()
    with torch.no_grad():
        ft_val_losses.append(criterion(ft_model(X_val), y_val).item())

    print(f"Epoch {epoch + 1}/{n_epochs} - train: {ft_train_losses[-1]:.4f} - val: {ft_val_losses[-1]:.4f}")

Grafiquemos juntos ambos entrenamientos:

In [ ]:
plt.plot(fe_val_losses, label="Feature extraction (val)")
plt.plot(ft_val_losses, label="Fine-tuning (val)")
plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("Feature extraction vs. fine-tuning")
plt.legend()
plt.show()

---

## 9. Comparar los tres enfoques

Evaluemos ambos en el conjunto de test intacto:

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

results = []
for name, model in [("Feature extraction", fe_model), ("Fine-tuning", ft_model)]:
    model.eval()
    with torch.no_grad():
        preds = (torch.sigmoid(model(X_test)) > 0.5).float()
    y_test_np = y_test.numpy().ravel()
    preds_np = preds.numpy().ravel()
    results.append({
        "Strategy": name,
        "Test accuracy": accuracy_score(y_test_np, preds_np),
        "Test F1": f1_score(y_test_np, preds_np, zero_division=0),
    })

import pandas as pd
pd.DataFrame(results)

Un gráfico de barras hace que la comparación de accuracy se lea más fácilmente de un vistazo:

In [ ]:
results_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(results_df["Strategy"], results_df["Test accuracy"], color=["steelblue", "darkorange"])
ax.set_ylabel("Test accuracy")
ax.set_title("Feature extraction vs. fine-tuning")
ax.set_ylim(0, 1)
plt.show()


Accuracy y F1 resumen; una matriz de confusión para cada estrategia muestra *qué* errores comete cada una:

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, model) in zip(axes, [("Feature extraction", fe_model), ("Fine-tuning", ft_model)]):
    model.eval()
    with torch.no_grad():
        model_preds = (torch.sigmoid(model(X_test)) > 0.5).float().numpy().ravel()
    cm = confusion_matrix(y_test.numpy().ravel(), model_preds, labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=["Absent", "Present"]).plot(cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()


**Pruébalo tú mismo**: mira predicciones reales junto a las imágenes de test reales (la misma comprobación cualitativa que usó `NB13`) — ¿parecen razonables a simple vista los errores del modelo con fine-tuning?

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, i in zip(axes.ravel(), range(8)):
    img_show = X_test[i].permute(1, 2, 0).numpy()
    img_show = (img_show - img_show.min()) / (img_show.max() - img_show.min())  # undo ImageNet normalization for display
    ax.imshow(img_show)
    ax.set_title(f"True: {int(y_test_np[i])}  Pred: {int(preds_np[i])}")
    ax.axis("off")
plt.suptitle("Fine-tuned model -- real predictions on real test images")
plt.tight_layout()
plt.show()


**Construye tú mismo la comparación completa**: añade una tercera fila a la tabla anterior con la accuracy de test de `NB13` para el `HullCNN` entrenado desde cero (vuelve a ejecutar ese notebook si no tienes la cifra a mano). Con solo ~240 imágenes de entrenamiento, ¿qué enfoque ha ganado aquí realmente — y coincide eso con la expectativa general de que el transfer learning ayuda más cuando tu propio dataset es pequeño en relación con lo que vio el modelo preentrenado? Si la extracción de características y el fine-tuning acaban muy cerca uno del otro, ese es también un resultado razonable: las características preentrenadas de ImageNet en `layer4` puede que ya fueran "suficientemente buenas" para esta tarea, dejando poco margen para que la flexibilidad extra del fine-tuning ayude con estos pocos datos — un resultado legítimo e informativo, no un fallo del notebook (el mismo planteamiento honesto que usó `NB14` para su propia comparación de referencia).

---

## Resumen de la clase

- El transfer learning reutiliza como punto de partida una red ya entrenada con un dataset grande y general (ResNet18 sobre ImageNet), en vez de aprenderlo todo desde cero.
- Funciona porque las primeras capas de una CNN aprenden características genéricas (bordes, texturas) útiles en casi cualquier tarea de imagen — la visualización de mapas de características de `NB13` lo hizo concreto antes de que nos apoyáramos en ello aquí.
- La extracción de características congela todo el backbone y entrena solo una nueva cabeza — rápida, con pocos parámetros entrenables, buena para datasets muy pequeños.
- El fine-tuning también descongela algunas capas finales del backbone, normalmente con una tasa de aprendizaje menor — más flexible, necesita más datos para que compense con seguridad.
- Los modelos preentrenados esperan un pipeline de preprocesado específico (`weights.transforms()`) — darles píxeles crudos 0-1 al estilo `NB13` perjudicaría el rendimiento silenciosamente.
- Hemos comparado la extracción de características, el fine-tuning y (por referencia) el entrenamiento desde cero de `NB13` sobre la misma tarea real, en vez de fiarnos de un único número de forma aislada.

## Para la próxima clase (NB16)

**Autoencoders y detección de anomalías** con Deep Learning — la contrapartida no supervisada de los clasificadores de este bloque, aprendiendo a reconstruir datos "normales" lo bastante bien como para que cualquier cosa mal reconstruida destaque como anómala.

## Tarea / Ideas de práctica

1. Descongela también `layer3` además de `layer4` en la Parte 8 y vuelve a entrenar — ¿mejora la accuracy de test, o la flexibilidad extra empieza a sobreajustar con este dataset pequeño?
2. Prueba un backbone preentrenado distinto, `models.resnet34(weights=ResNet34_Weights.DEFAULT)` (mismo patrón, distinta clase de pesos) — ¿ayuda aquí una red preentrenada más grande, o solo ralentiza el entrenamiento?
3. En la Parte 7, sube la tasa de aprendizaje de `0.001` a `0.01` — ¿sigue entrenando de forma estable la extracción de características? Ahora prueba el mismo cambio en el optimizador de fine-tuning de la Parte 8 — ¿sigue *ese* entrenando de forma estable? ¿Qué te dice la diferencia sobre por qué el fine-tuning usa por convención una tasa de aprendizaje menor?
4. Usando el objeto `preprocess` impreso en la Parte 6, averigua las dimensiones de redimensionado y los valores de normalización que aplica realmente — ¿coinciden con la descripción "típicamente 224×224, media/std de ImageNet" de esa sección?
5. Vuelve a ejecutar la comparación de la Parte 9 usando `weather_conditions` u otra clase objetivo de la lista de máscaras de `NB13` en vez de crecimiento marino — ¿cambia la clasificación relativa de los tres enfoques?

> ***Como siempre: cuando dos variantes de modelo obtienen resultados muy cercanos, la tarea más útil es explicar *por qué* están cerca, no solo elegir el número que sea marginalmente mayor.***